# Image Captioning — CNN Encoder + Bahdanau Attention + LSTM Decoder
### Phase 1 — Notebook (run cells top to bottom, in order)

**Order:** Setup → Download data → Clean captions/build vocab → Dataset/DataLoader → Model →
Metrics → Training → Inference → Attention visualization.


## 1. Setup & Config

In [1]:
import os, sys, json, math, time, pickle, random, re, string, zipfile, urllib.request
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field, asdict

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torchvision
from torchvision import transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce GTX 1650 Ti


In [2]:
@dataclass
class Config:
    # paths
    data_root: str = "data/coco"
    train_images: str = "data/coco/train2017"
    val_images: str = "data/coco/val2017"
    train_ann: str = "data/coco/annotations/captions_train2017.json"
    val_ann: str = "data/coco/annotations/captions_val2017.json"
    cache_dir: str = "data/coco/cache"
    checkpoint_dir: str = "checkpoints"
    output_dir: str = "outputs/predictions"

    # vocab
    min_word_freq: int = 5
    max_caption_len: int = 50

    # model
    encoder_name: str = "efficientnet_b4"
    encoder_dim: int = 1792
    attention_dim: int = 512
    embed_dim: int = 512
    decoder_dim: int = 512
    dropout: float = 0.5
    fine_tune_encoder_from_epoch: int = 10

    # training
    batch_size: int = 32
    epochs: int = 50
    lr_encoder: float = 1e-4
    lr_decoder: float = 4e-4
    weight_decay: float = 1e-5
    grad_clip: float = 5.0
    teacher_forcing_ratio: float = 1.0 
    scheduled_sampling_decay: float = 0.98
    early_stopping_patience: int = 15
    amp: bool = True
    num_workers: int = 4
    checkpoint_every_n_batches: int = 2000   # mid-epoch safety checkpoint

    # inference
    beam_sizes: tuple = (3, 5, 7, 10)

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()
for d in [cfg.cache_dir, cfg.checkpoint_dir, cfg.output_dir]:
    os.makedirs(d, exist_ok=True)
print(asdict(cfg))


{'data_root': 'data/coco', 'train_images': 'data/coco/train2017', 'val_images': 'data/coco/val2017', 'train_ann': 'data/coco/annotations/captions_train2017.json', 'val_ann': 'data/coco/annotations/captions_val2017.json', 'cache_dir': 'data/coco/cache', 'checkpoint_dir': 'checkpoints', 'output_dir': 'outputs/predictions', 'min_word_freq': 5, 'max_caption_len': 50, 'encoder_name': 'efficientnet_b4', 'encoder_dim': 1792, 'attention_dim': 512, 'embed_dim': 512, 'decoder_dim': 512, 'dropout': 0.5, 'fine_tune_encoder_from_epoch': 10, 'batch_size': 32, 'epochs': 50, 'lr_encoder': 0.0001, 'lr_decoder': 0.0004, 'weight_decay': 1e-05, 'grad_clip': 5.0, 'teacher_forcing_ratio': 1.0, 'scheduled_sampling_decay': 0.98, 'early_stopping_patience': 15, 'amp': True, 'num_workers': 4, 'checkpoint_every_n_batches': 2000, 'beam_sizes': (3, 5, 7, 10), 'device': 'cuda'}


## 2. Dataset Download (Windows)

**PowerShell:**
```powershell
mkdir data\coco; cd data\coco
Invoke-WebRequest -Uri http://images.cocodataset.org/zips/train2017.zip -OutFile train2017.zip
Invoke-WebRequest -Uri http://images.cocodataset.org/zips/val2017.zip -OutFile val2017.zip
Invoke-WebRequest -Uri http://images.cocodataset.org/annotations/annotations_trainval2017.zip -OutFile annotations_trainval2017.zip
Expand-Archive train2017.zip -DestinationPath .
Expand-Archive val2017.zip -DestinationPath .
Expand-Archive annotations_trainval2017.zip -DestinationPath .
```
Or run the cell below (same effect, works cross-platform, shows progress). Skip this cell entirely once `data/coco/` is already populated.


In [3]:
COCO_URLS = {
    "train2017.zip": "http://images.cocodataset.org/zips/train2017.zip",
    "val2017.zip": "http://images.cocodataset.org/zips/val2017.zip",
    "annotations_trainval2017.zip": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
}

def _progress(block_num, block_size, total_size):
    downloaded = block_num * block_size
    pct = min(100, downloaded * 100 / total_size) if total_size > 0 else 0
    print(f"\r  {pct:5.1f}%", end="")

def download_coco(dest_dir=cfg.data_root, which=("train2017.zip", "val2017.zip", "annotations_trainval2017.zip")):
    os.makedirs(dest_dir, exist_ok=True)
    for fname in which:
        url = COCO_URLS[fname]
        out_path = os.path.join(dest_dir, fname)
        if os.path.exists(out_path):
            print(f"[skip] {fname} already downloaded")
        else:
            print(f"Downloading {fname} ...")
            urllib.request.urlretrieve(url, out_path, _progress)
            print()
        print(f"Extracting {fname} ...")
        with zipfile.ZipFile(out_path, "r") as zf:
            zf.extractall(dest_dir)
    print("Done. COCO 2017 ready at", dest_dir)

# Uncomment to download (skip if data/coco/ is already populated):
download_coco()


    0.1%

KeyboardInterrupt: 

## 3. Caption Cleaning, Vocabulary & `.pkl` Caching

In [5]:
PUNCT_TABLE = str.maketrans("", "", string.punctuation)

def clean_caption(caption: str) -> list:
    caption = caption.lower().strip()
    caption = caption.translate(PUNCT_TABLE)
    caption = re.sub(r"\s+", " ", caption).strip()
    return caption.split()

def load_coco_annotations(ann_path):
    with open(ann_path, "r") as f:
        data = json.load(f)
    id2file = {img["id"]: img["file_name"] for img in data["images"]}
    id2caps = {}
    for ann in data["annotations"]:
        id2caps.setdefault(ann["image_id"], []).append(ann["caption"])
    return id2caps, id2file


class Vocabulary:
    PAD, START, END, UNK = "<pad>", "<start>", "<end>", "<unk>"

    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}

    def build(self, all_tokenized_captions, min_freq=5):
        counter = Counter()
        for toks in all_tokenized_captions:
            counter.update(toks)
        specials = [self.PAD, self.START, self.END, self.UNK]
        words = specials + [w for w, c in counter.items() if c >= min_freq and w not in specials]
        self.word2idx = {w: i for i, w in enumerate(words)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        return self

    def encode(self, tokens, max_len):
        ids = [self.word2idx.get(self.START)]
        ids += [self.word2idx.get(t, self.word2idx[self.UNK]) for t in tokens[: max_len - 2]]
        ids.append(self.word2idx[self.END])
        return ids

    def decode(self, ids, join=True):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), self.UNK)
            if w == self.END:
                break
            if w in (self.START, self.PAD):
                continue
            words.append(w)
        return " ".join(words) if join else words

    def __len__(self):
        return len(self.word2idx)


def build_or_load_vocab_and_captions(ann_path, cache_dir, split_name, min_freq):
    vocab_path = os.path.join(cache_dir, "vocab.pkl")
    caps_path = os.path.join(cache_dir, f"cleaned_captions_{split_name}.pkl")
    paths_path = os.path.join(cache_dir, f"image_paths_{split_name}.pkl")

    if os.path.exists(caps_path) and os.path.exists(paths_path):
        print(f"[cache hit] loading cleaned captions for {split_name}")
        with open(caps_path, "rb") as f:
            id2clean = pickle.load(f)
        with open(paths_path, "rb") as f:
            id2file = pickle.load(f)
    else:
        print(f"[cache miss] cleaning captions for {split_name} ...")
        id2caps, id2file = load_coco_annotations(ann_path)
        id2clean = {img_id: [clean_caption(c) for c in caps] for img_id, caps in id2caps.items()}
        with open(caps_path, "wb") as f:
            pickle.dump(id2clean, f)
        with open(paths_path, "wb") as f:
            pickle.dump(id2file, f)

    if split_name == "train":
        if os.path.exists(vocab_path):
            print("[cache hit] loading vocab.pkl")
            with open(vocab_path, "rb") as f:
                vocab = pickle.load(f)
        else:
            print("[cache miss] building vocabulary ...")
            all_tok = [toks for caps in id2clean.values() for toks in caps]
            vocab = Vocabulary().build(all_tok, min_freq=min_freq)
            with open(vocab_path, "wb") as f:
                pickle.dump(vocab, f)
        return id2clean, id2file, vocab

    return id2clean, id2file


In [7]:
train_id2clean, train_id2file, vocab = build_or_load_vocab_and_captions(
    cfg.train_ann, cfg.cache_dir, "train", cfg.min_word_freq
)
val_id2clean, val_id2file = build_or_load_vocab_and_captions(
    cfg.val_ann, cfg.cache_dir, "val", cfg.min_word_freq
)
print("Vocab size:", len(vocab))


[cache hit] loading cleaned captions for train


[cache hit] loading vocab.pkl
[cache hit] loading cleaned captions for val
Vocab size: 10307


## 4. Dataset & DataLoader

In [8]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CocoCaptionDataset(Dataset):
    def __init__(self, images_dir, id2clean, id2file, vocab: Vocabulary, max_len, transform):
        self.images_dir = images_dir
        self.vocab = vocab
        self.max_len = max_len
        self.transform = transform
        self.samples = []
        for img_id, caps in id2clean.items():
            if img_id not in id2file:
                continue
            for toks in caps:
                self.samples.append((img_id, toks))
        self.id2file = id2file

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_id, tokens = self.samples[idx]
        img_path = os.path.join(self.images_dir, self.id2file[img_id])
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        caption_ids = self.vocab.encode(tokens, self.max_len)
        return image, torch.tensor(caption_ids, dtype=torch.long), img_id


def collate_fn(batch):
    images, captions, img_ids = zip(*batch)
    images = torch.stack(images, dim=0)
    lengths = [len(c) for c in captions]
    max_len = max(lengths)
    padded = torch.full((len(captions), max_len), vocab.word2idx[Vocabulary.PAD], dtype=torch.long)
    for i, c in enumerate(captions):
        padded[i, : len(c)] = c
    lengths = torch.tensor(lengths, dtype=torch.long)
    return images, padded, lengths, list(img_ids)


train_dataset = CocoCaptionDataset(cfg.train_images, train_id2clean, train_id2file, vocab, cfg.max_caption_len, train_transform)
val_dataset = CocoCaptionDataset(cfg.val_images, val_id2clean, val_id2file, vocab, cfg.max_caption_len, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
                           num_workers=cfg.num_workers, collate_fn=collate_fn, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers, collate_fn=collate_fn, pin_memory=True)

print("Train samples (image,caption pairs):", len(train_dataset))
print("Val samples:", len(val_dataset))


Train samples (image,caption pairs): 591753
Val samples: 25014


## 5. Model — EfficientNet-B4 Encoder + Bahdanau Attention + LSTM Decoder

In [9]:
class EncoderCNN(nn.Module):
    def __init__(self, encoded_image_size=7, fine_tune=False):
        super().__init__()
        self.enc_image_size = encoded_image_size
        backbone = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        self.adaptive_pool = nn.AdaptiveAvgPool2d((encoded_image_size, encoded_image_size))
        self.fine_tune(fine_tune)

    def forward(self, images):
        feats = self.features(images)
        feats = self.adaptive_pool(feats)
        feats = feats.permute(0, 2, 3, 1)
        return feats

    def fine_tune(self, fine_tune=True):
        for p in self.features.parameters():
            p.requires_grad = False
        if fine_tune:
            for block in list(self.features.children())[-4:]:
                for p in block.parameters():
                    p.requires_grad = True


class BahdanauAttention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)
        self.tanh = nn.Tanh()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, encoder_out, decoder_hidden):
        att1 = self.encoder_att(encoder_out)
        att2 = self.decoder_att(decoder_hidden).unsqueeze(1)
        att = self.full_att(self.tanh(att1 + att2)).squeeze(2)
        alpha = self.softmax(att)
        context = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)
        return context, alpha


class DecoderWithAttention(nn.Module):
    def __init__(self, attention_dim, embed_dim, decoder_dim, vocab_size, encoder_dim, dropout=0.5):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.attention_dim = attention_dim
        self.embed_dim = embed_dim
        self.decoder_dim = decoder_dim
        self.vocab_size = vocab_size

        self.attention = BahdanauAttention(encoder_dim, decoder_dim, attention_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout = nn.Dropout(p=dropout)
        self.lstm_cell = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim, bias=True)

        self.init_h = nn.Linear(encoder_dim, decoder_dim)
        self.init_c = nn.Linear(encoder_dim, decoder_dim)
        self.f_beta = nn.Linear(decoder_dim, encoder_dim)
        self.sigmoid = nn.Sigmoid()
        self.fc = nn.Linear(decoder_dim, vocab_size)

    def init_hidden_state(self, encoder_out):
        mean_enc = encoder_out.mean(dim=1)
        h = self.init_h(mean_enc)
        c = self.init_c(mean_enc)
        return h, c

    def forward(self, encoder_out, captions, lengths, teacher_forcing_ratio=1.0):
        batch_size = encoder_out.size(0)
        encoder_out = encoder_out.view(batch_size, -1, self.encoder_dim)
        num_pixels = encoder_out.size(1)

        h, c = self.init_hidden_state(encoder_out)

        decode_lengths = (lengths - 1).tolist()
        max_decode_len = max(decode_lengths)

        predictions = torch.zeros(batch_size, max_decode_len, self.vocab_size, device=encoder_out.device)
        alphas = torch.zeros(batch_size, max_decode_len, num_pixels, device=encoder_out.device)

        embeddings = self.embedding(captions)
        prev_word_embed = embeddings[:, 0, :]

        for t in range(max_decode_len):
            batch_size_t = sum([l > t for l in decode_lengths])
            context, alpha = self.attention(encoder_out[:batch_size_t], h[:batch_size_t])
            gate = self.sigmoid(self.f_beta(h[:batch_size_t]))
            context = gate * context

            lstm_input = torch.cat([prev_word_embed[:batch_size_t], context], dim=1)
            h_t, c_t = self.lstm_cell(lstm_input, (h[:batch_size_t], c[:batch_size_t]))
            h = torch.cat([h_t, h[batch_size_t:]], dim=0) if batch_size_t < batch_size else h_t
            c = torch.cat([c_t, c[batch_size_t:]], dim=0) if batch_size_t < batch_size else c_t

            preds = self.fc(self.dropout(h_t))
            predictions[:batch_size_t, t, :] = preds
            alphas[:batch_size_t, t, :] = alpha

            use_teacher = random.random() < teacher_forcing_ratio
            if use_teacher and t + 1 < embeddings.size(1):
                next_embed = embeddings[:, t + 1, :]
            else:
                next_ids = preds.argmax(dim=1)
                next_embed_t = self.embedding(next_ids)
                if next_embed_t.size(0) < embeddings.size(0):
                    pad = self.embedding(
                        torch.zeros(embeddings.size(0) - next_embed_t.size(0), dtype=torch.long, device=embeddings.device)
                    )
                    next_embed_t = torch.cat([next_embed_t, pad], dim=0)
                next_embed = next_embed_t
            prev_word_embed = next_embed

        return predictions, alphas, decode_lengths


In [ ]:
encoder = EncoderCNN(fine_tune=False).to(device)
decoder = DecoderWithAttention(
    attention_dim=cfg.attention_dim,
    embed_dim=cfg.embed_dim,
    decoder_dim=cfg.decoder_dim,
    vocab_size=len(vocab),
    encoder_dim=cfg.encoder_dim,
    dropout=cfg.dropout,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=vocab.word2idx[Vocabulary.PAD])

decoder_optimizer = torch.optim.AdamW(decoder.parameters(), lr=cfg.lr_decoder, weight_decay=cfg.weight_decay)
encoder_optimizer = None  # created once the encoder is unfrozen (epoch >= cfg.fine_tune_encoder_from_epoch)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(decoder_optimizer, T_max=cfg.epochs)
scaler = GradScaler(device.type, enabled=cfg.amp)


## 6. Metrics — BLEU-1..4, METEOR, ROUGE-L, CIDEr-D

Must be run **before** training — the training loop calls `evaluate()` at the end of every epoch.


In [10]:
# Install once: pip install nltk rouge-score pycocoevalcap
import nltk
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet")

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

smoothie = SmoothingFunction().method4
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

try:
    from pycocoevalcap.cider.cider import Cider
    _cider_scorer = Cider()
    HAS_CIDER = True
except Exception as e:
    print("pycocoevalcap not available, CIDEr-D will be skipped:", e)
    HAS_CIDER = False


@torch.no_grad()
def generate_greedy(encoder_out, max_len=cfg.max_caption_len):
    encoder_out = encoder_out.view(1, -1, cfg.encoder_dim)
    h, c = decoder.init_hidden_state(encoder_out)
    word = torch.tensor([vocab.word2idx[Vocabulary.START]], device=device)
    words, alphas = [], []
    for _ in range(max_len):
        embed = decoder.embedding(word)
        context, alpha = decoder.attention(encoder_out, h)
        gate = decoder.sigmoid(decoder.f_beta(h))
        context = gate * context
        h, c = decoder.lstm_cell(torch.cat([embed, context], dim=1), (h, c))
        preds = decoder.fc(h)
        word = preds.argmax(dim=1)
        alphas.append(alpha.squeeze(0).cpu().numpy())
        if word.item() == vocab.word2idx[Vocabulary.END]:
            break
        words.append(word.item())
    return words, alphas


@torch.no_grad()
def evaluate(loader, max_batches=None):
    encoder.eval(); decoder.eval()
    refs_by_id, hyps_by_id = {}, {}

    for bi, (images, captions, lengths, img_ids) in enumerate(loader):
        if max_batches and bi >= max_batches:
            break
        images = images.to(device)
        enc_out = encoder(images)
        for i in range(images.size(0)):
            words, _ = generate_greedy(enc_out[i:i+1])
            hyp = vocab.decode(words, join=False)
            img_id = img_ids[i]
            hyps_by_id.setdefault(img_id, hyp)
            gt_tokens = vocab.decode(captions[i].tolist(), join=False)
            refs_by_id.setdefault(img_id, []).append(gt_tokens)

    bleu1_l, bleu2_l, bleu3_l, bleu4_l, meteor_l, rougeL_l = [], [], [], [], [], []
    for img_id, hyp in hyps_by_id.items():
        refs = refs_by_id[img_id]
        if len(hyp) == 0:
            continue
        bleu1_l.append(sentence_bleu(refs, hyp, weights=(1, 0, 0, 0), smoothing_function=smoothie))
        bleu2_l.append(sentence_bleu(refs, hyp, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie))
        bleu3_l.append(sentence_bleu(refs, hyp, weights=(1/3, 1/3, 1/3, 0), smoothing_function=smoothie))
        bleu4_l.append(sentence_bleu(refs, hyp, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie))
        try:
            meteor_l.append(meteor_score(refs, hyp))
        except Exception:
            pass
        hyp_str, ref_str = " ".join(hyp), " ".join(refs[0])
        rougeL_l.append(rouge.score(ref_str, hyp_str)["rougeL"].fmeasure)

    metrics = {
        "BLEU-1": float(np.mean(bleu1_l)) if bleu1_l else 0.0,
        "BLEU-2": float(np.mean(bleu2_l)) if bleu2_l else 0.0,
        "BLEU-3": float(np.mean(bleu3_l)) if bleu3_l else 0.0,
        "BLEU-4": float(np.mean(bleu4_l)) if bleu4_l else 0.0,
        "METEOR": float(np.mean(meteor_l)) if meteor_l else 0.0,
        "ROUGE-L": float(np.mean(rougeL_l)) if rougeL_l else 0.0,
    }

    if HAS_CIDER:
        gts = {str(k): [" ".join(r) for r in v] for k, v in refs_by_id.items()}
        res = {str(k): [" ".join(v)] for k, v in hyps_by_id.items()}
        cider_score, _ = _cider_scorer.compute_score(gts, res)
        metrics["CIDEr-D"] = float(cider_score)

    return metrics


[nltk_data] Downloading package wordnet to /home/engadm/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 7. Training — AdamW + CosineAnnealingLR + AMP + Grad Clipping + Checkpointing

Saves `last_model.pth` every epoch **and** every `cfg.checkpoint_every_n_batches` batches mid-epoch,
so a crash never costs more than a partial epoch.


In [ ]:
def save_checkpoint(state, is_best, checkpoint_dir=cfg.checkpoint_dir):
    torch.save(state, os.path.join(checkpoint_dir, "last_model.pth"))
    if is_best:
        torch.save(state, os.path.join(checkpoint_dir, "best_model.pth"))


def current_state(epoch, best_bleu4):
    return {
        "epoch": epoch,
        "encoder": encoder.state_dict(),
        "decoder": decoder.state_dict(),
        "decoder_optimizer": decoder_optimizer.state_dict(),
        "encoder_optimizer": encoder_optimizer.state_dict() if encoder_optimizer else None,
        "best_bleu4": best_bleu4,
        "vocab": vocab,
        "config": asdict(cfg),
    }


def load_checkpoint(path, encoder, decoder, decoder_optimizer, encoder_optimizer=None):
    state = torch.load(path, map_location=device, weights_only=False)
    encoder.load_state_dict(state["encoder"])
    decoder.load_state_dict(state["decoder"])
    decoder_optimizer.load_state_dict(state["decoder_optimizer"])
    if encoder_optimizer is not None and state.get("encoder_optimizer") is not None:
        encoder_optimizer.load_state_dict(state["encoder_optimizer"])
    return state["epoch"], state.get("best_bleu4", 0.0)


def train_one_epoch(epoch, teacher_forcing_ratio, best_bleu4_so_far):
    encoder.train(); decoder.train()
    running_loss, n_batches = 0.0, 0
    for i, (images, captions, lengths, _) in enumerate(train_loader):
        images, captions, lengths = images.to(device), captions.to(device), lengths.to(device)

        decoder_optimizer.zero_grad()
        if encoder_optimizer is not None:
            encoder_optimizer.zero_grad()

        with autocast(device.type, enabled=cfg.amp):
            encoder_out = encoder(images)
            predictions, alphas, decode_lengths = decoder(encoder_out, captions, lengths, teacher_forcing_ratio)
            targets = captions[:, 1:]
            packed_preds = torch.cat([predictions[b, :decode_lengths[b], :] for b in range(len(decode_lengths))], dim=0)
            packed_targets = torch.cat([targets[b, :decode_lengths[b]] for b in range(len(decode_lengths))], dim=0)
            loss = criterion(packed_preds, packed_targets)
            loss = loss + 1.0 * ((1.0 - alphas.sum(dim=1)) ** 2).mean()

        scaler.scale(loss).backward()
        scaler.unscale_(decoder_optimizer)
        nn.utils.clip_grad_norm_(decoder.parameters(), cfg.grad_clip)
        if encoder_optimizer is not None:
            scaler.unscale_(encoder_optimizer)
            nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, encoder.parameters()), cfg.grad_clip)

        scaler.step(decoder_optimizer)
        if encoder_optimizer is not None:
            scaler.step(encoder_optimizer)
        scaler.update()

        running_loss += loss.item()
        n_batches += 1
        if i % 100 == 0:
            print(f"  epoch {epoch} batch {i}/{len(train_loader)} loss {loss.item():.4f}")

        # mid-epoch safety checkpoint — protects against losing a full epoch to a late crash
        if cfg.checkpoint_every_n_batches and i > 0 and i % cfg.checkpoint_every_n_batches == 0:
            save_checkpoint(current_state(epoch, best_bleu4_so_far), is_best=False)

    return running_loss / max(n_batches, 1)


def run_training(resume=False, epochs=None):
    global encoder_optimizer
    epochs = epochs or cfg.epochs
    start_epoch, best_bleu4, patience_counter = 0, 0.0, 0
    tf_ratio = cfg.teacher_forcing_ratio

    last_ckpt = os.path.join(cfg.checkpoint_dir, "last_model.pth")
    if resume and os.path.exists(last_ckpt):
        state = torch.load(last_ckpt, map_location=device, weights_only=False)
        if state.get("encoder_optimizer") is not None:
            # only build the encoder optimizer if it was already unfrozen when this checkpoint was saved
            encoder.fine_tune(True)
            encoder_optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, encoder.parameters()), lr=cfg.lr_encoder, weight_decay=cfg.weight_decay
            )
        start_epoch, best_bleu4 = load_checkpoint(last_ckpt, encoder, decoder, decoder_optimizer, encoder_optimizer)
        start_epoch += 1
        print(f"Resumed from epoch {start_epoch}, best BLEU-4 so far: {best_bleu4:.4f}")

    for epoch in range(start_epoch, epochs):
        if epoch == cfg.fine_tune_encoder_from_epoch and encoder_optimizer is None:
            print(">> Unfreezing encoder for fine-tuning")
            encoder.fine_tune(True)
            encoder_optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, encoder.parameters()), lr=cfg.lr_encoder, weight_decay=cfg.weight_decay
            )

        t0 = time.time()
        train_loss = train_one_epoch(epoch, tf_ratio, best_bleu4)
        val_metrics = evaluate(val_loader, max_batches=50)
        scheduler.step()
        tf_ratio *= cfg.scheduled_sampling_decay

        bleu4 = val_metrics["BLEU-4"]
        is_best = bleu4 > best_bleu4
        best_bleu4 = max(bleu4, best_bleu4)
        patience_counter = 0 if is_best else patience_counter + 1

        print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_BLEU4={bleu4:.4f} "
              f"(best={best_bleu4:.4f}) tf_ratio={tf_ratio:.3f} time={time.time()-t0:.1f}s")

        save_checkpoint(current_state(epoch, best_bleu4), is_best)

        if patience_counter >= cfg.early_stopping_patience:
            print(f"Early stopping at epoch {epoch} (no improvement for {cfg.early_stopping_patience} epochs)")
            break

# Start training:
# run_training()
# Resume after a crash/interruption:
run_training(resume=True)


Resumed from epoch 1, best BLEU-4 so far: 0.0000
  epoch 1 batch 0/18492 loss 3.5157
  epoch 1 batch 100/18492 loss 3.9666
  epoch 1 batch 200/18492 loss 4.2350
  epoch 1 batch 300/18492 loss 4.1684
  epoch 1 batch 400/18492 loss 3.9111
  epoch 1 batch 500/18492 loss 4.2890
  epoch 1 batch 600/18492 loss 3.7362
  epoch 1 batch 700/18492 loss 3.8261
  epoch 1 batch 800/18492 loss 3.8668
  epoch 1 batch 900/18492 loss 4.3221
  epoch 1 batch 1000/18492 loss 3.8081
  epoch 1 batch 1100/18492 loss 3.9542
  epoch 1 batch 1200/18492 loss 4.1781
  epoch 1 batch 1300/18492 loss 4.1530
  epoch 1 batch 1400/18492 loss 3.7310
  epoch 1 batch 1500/18492 loss 3.5192
  epoch 1 batch 1600/18492 loss 3.8062
  epoch 1 batch 1700/18492 loss 3.9747
  epoch 1 batch 1800/18492 loss 3.6568
  epoch 1 batch 1900/18492 loss 4.0962
  epoch 1 batch 2000/18492 loss 4.2340
  epoch 1 batch 2100/18492 loss 4.1212
  epoch 1 batch 2200/18492 loss 3.8322
  epoch 1 batch 2300/18492 loss 3.7035
  epoch 1 batch 2400/18492 

## 8. Inference — Greedy Decoding & Beam Search

In [12]:
@torch.no_grad()
def beam_search_caption(image_tensor, beam_size=5, max_len=cfg.max_caption_len):
    encoder.eval(); decoder.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)
    enc_out = encoder(image_tensor).view(1, -1, cfg.encoder_dim)
    num_pixels = enc_out.size(1)

    k = beam_size
    enc_out = enc_out.expand(k, num_pixels, cfg.encoder_dim)

    start_id = vocab.word2idx[Vocabulary.START]
    end_id = vocab.word2idx[Vocabulary.END]

    seqs = torch.full((k, 1), start_id, dtype=torch.long, device=device)
    top_k_scores = torch.zeros(k, device=device)
    seq_alphas = torch.zeros(k, 1, enc_out.size(1), device=device)

    h, c = decoder.init_hidden_state(enc_out)
    complete_seqs, complete_scores, complete_alphas = [], [], []

    step = 1
    k_prev_words = torch.full((k, 1), start_id, dtype=torch.long, device=device)

    while True:
        embed = decoder.embedding(k_prev_words).squeeze(1)
        context, alpha = decoder.attention(enc_out, h)
        gate = decoder.sigmoid(decoder.f_beta(h))
        context = gate * context
        h, c = decoder.lstm_cell(torch.cat([embed, context], dim=1), (h, c))
        scores = F.log_softmax(decoder.fc(h), dim=1)
        scores = top_k_scores.unsqueeze(1).expand_as(scores) + scores

        if step == 1:
            top_k_scores, top_k_words = scores[0].topk(k, 0, True, True)
        else:
            top_k_scores, top_k_words = scores.view(-1).topk(k, 0, True, True)

        prev_word_inds = top_k_words // len(vocab)
        next_word_inds = top_k_words % len(vocab)

        seqs = torch.cat([seqs[prev_word_inds], next_word_inds.unsqueeze(1)], dim=1)
        seq_alphas = torch.cat([seq_alphas[prev_word_inds], alpha[prev_word_inds].unsqueeze(1)], dim=1)

        incomplete = [i for i, w in enumerate(next_word_inds) if w.item() != end_id]
        complete = [i for i, w in enumerate(next_word_inds) if w.item() == end_id]

        if complete:
            complete_seqs.extend(seqs[complete].tolist())
            complete_scores.extend(top_k_scores[complete].tolist())
            complete_alphas.extend(seq_alphas[complete].tolist())
        k -= len(complete)
        if k == 0 or step >= max_len:
            break

        seqs = seqs[incomplete]
        seq_alphas = seq_alphas[incomplete]
        h, c = h[prev_word_inds[incomplete]], c[prev_word_inds[incomplete]]
        enc_out = enc_out[prev_word_inds[incomplete]]
        top_k_scores = top_k_scores[incomplete]
        k_prev_words = next_word_inds[incomplete].unsqueeze(1)
        step += 1

    if not complete_scores:
        complete_seqs = seqs.tolist()
        complete_scores = top_k_scores.tolist()
        complete_alphas = seq_alphas.tolist()

    best_idx = int(np.argmax(complete_scores))
    best_seq = complete_seqs[best_idx][1:]
    best_alphas = complete_alphas[best_idx][1:]
    caption = vocab.decode(best_seq)
    return caption, best_seq, best_alphas


def predict_image(image_path, beam_size=5, visualize=False):
    image = Image.open(image_path).convert("RGB")
    tensor = eval_transform(image)
    caption, ids, alphas = beam_search_caption(tensor, beam_size=beam_size)
    print("-" * 60)
    print(f"  Image   : {image_path}")
    print(f"  Mode    : beam_k{beam_size}")
    print(f"  Caption : {caption}")
    print("-" * 60)
    if visualize:
        visualize_attention(image, ids, alphas, image_path, beam_size)
    return caption


## 9. Attention Visualization

In [13]:
import matplotlib.pyplot as plt
from scipy.ndimage import zoom as ndi_zoom

def visualize_attention(image: Image.Image, word_ids, alphas, image_path, beam_size, grid=7):
    words = [vocab.idx2word.get(i, Vocabulary.UNK) for i in word_ids]
    image = image.resize((336, 336))
    n = len(words)
    cols = min(5, max(1, n))
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.array(axes).reshape(-1)

    for t, word in enumerate(words):
        ax = axes[t]
        ax.imshow(image)
        alpha_map = np.array(alphas[t]).reshape(grid, grid)
        alpha_img = ndi_zoom(alpha_map, 336 / grid, order=3)
        ax.imshow(alpha_img, alpha=0.6, cmap="jet")
        ax.set_title(word, fontsize=10)
        ax.axis("off")
    for ax in axes[len(words):]:
        ax.axis("off")

    plt.tight_layout()
    stem = Path(image_path).stem
    out_path = os.path.join(cfg.output_dir, f"{stem}_beam_k{beam_size}_attention.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Attention map saved →", out_path)

    fig2, ax2 = plt.subplots(figsize=(5, 5))
    ax2.imshow(image); ax2.axis("off")
    ax2.set_title(" ".join(words), fontsize=11, wrap=True)
    cap_path = os.path.join(cfg.output_dir, f"{stem}_beam_k{beam_size}_caption.png")
    plt.savefig(cap_path, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print("Caption figure saved →", cap_path)
